In [1]:
import pandas as pd
import numpy as np
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('paysim.csv/PS_20174392719_1491204439457_log.csv')

# Keep only TRANSFER and CASH_OUT — proven in EDA that fraud only exists here
df_focus = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

print(f"Full dataset: {len(df):,} transactions")
print(f"Working dataset (TRANSFER + CASH_OUT): {len(df_focus):,} transactions")
print(f"Fraud cases in working dataset: {df_focus['isFraud'].sum():,}")

Full dataset: 6,362,620 transactions
Working dataset (TRANSFER + CASH_OUT): 2,770,409 transactions
Fraud cases in working dataset: 8,213


In [2]:
print("Building account-level aggregations...")

# Group all transactions by sender account
account_stats = df_focus.groupby('nameOrig').agg(
    total_transactions    = ('amount', 'count'),
    total_sent            = ('amount', 'sum'),
    avg_amount            = ('amount', 'mean'),
    max_amount            = ('amount', 'max'),
    total_received        = ('oldbalanceOrg', 'first'),
    unique_destinations   = ('nameDest', 'nunique'),
    unique_sources        = ('nameOrig', 'count'),
    first_active_step     = ('step', 'min'),
    last_active_step      = ('step', 'max'),
    balance_before_sum    = ('oldbalanceOrg', 'sum'),
    balance_after_sum     = ('newbalanceOrig', 'sum'),
    is_fraud              = ('isFraud', 'max')
).reset_index()

print(f"Total unique sender accounts: {len(account_stats):,}")
print(f"Fraud accounts: {account_stats['is_fraud'].sum():,}")
print(f"Legitimate accounts: {(account_stats['is_fraud']==0).sum():,}")
print("\nSample:")
print(account_stats.head(3))

Building account-level aggregations...
Total unique sender accounts: 2,768,630
Fraud accounts: 8,213
Legitimate accounts: 2,760,417

Sample:
      nameOrig  total_transactions  total_sent  avg_amount  max_amount  \
0  C1000000639                   1   244486.46   244486.46   244486.46   
1  C1000004053                   1   211189.64   211189.64   211189.64   
2  C1000004530                   1    93865.13    93865.13    93865.13   

   total_received  unique_destinations  unique_sources  first_active_step  \
0         8946.00                    1               1                249   
1            0.00                    1               1                327   
2       351422.72                    1               1                 41   

   last_active_step  balance_before_sum  balance_after_sum  is_fraud  
0               249             8946.00               0.00         0  
1               327                0.00               0.00         0  
2                41           351422.72 

In [3]:
# FEATURE 1: VELOCITY RATIO
# What proportion of received funds is forwarded out?
# Mule accounts forward everything — ratio close to 1.0

account_stats['velocity_ratio'] = account_stats.apply(
    lambda row: row['total_sent'] / row['balance_before_sum'] 
    if row['balance_before_sum'] > 0 else 1.0, axis=1
).clip(0, 1)

# Validate
fraud_vr = account_stats[account_stats['is_fraud']==1]['velocity_ratio']
legit_vr = account_stats[account_stats['is_fraud']==0]['velocity_ratio']

print("=== FEATURE 1: VELOCITY RATIO ===")
print(f"Fraud accounts   — mean velocity ratio: {fraud_vr.mean():.4f}")
print(f"Legitimate accounts — mean velocity ratio: {legit_vr.mean():.4f}")
print(f"Fraud accounts with ratio > 0.90: {(fraud_vr > 0.90).sum():,} ({(fraud_vr > 0.90).mean()*100:.1f}%)")
print(f"Legit accounts with ratio > 0.90: {(legit_vr > 0.90).sum():,} ({(legit_vr > 0.90).mean()*100:.1f}%)")

=== FEATURE 1: VELOCITY RATIO ===
Fraud accounts   — mean velocity ratio: 0.9925
Legitimate accounts — mean velocity ratio: 0.9459
Fraud accounts with ratio > 0.90: 8,074 (98.3%)
Legit accounts with ratio > 0.90: 2,507,743 (90.8%)


In [4]:
# FEATURE 2: BALANCE RETENTION RATE
# How much of received funds is still in the account after all transactions?
# Mule accounts retain almost nothing — rate close to 0.0

account_stats['balance_retention_rate'] = account_stats.apply(
    lambda row: row['balance_after_sum'] / row['balance_before_sum']
    if row['balance_before_sum'] > 0 else 0.0, axis=1
).clip(0, 1)

# Validate
fraud_brr = account_stats[account_stats['is_fraud']==1]['balance_retention_rate']
legit_brr = account_stats[account_stats['is_fraud']==0]['balance_retention_rate']

print("=== FEATURE 2: BALANCE RETENTION RATE ===")
print(f"Fraud accounts   — mean retention rate: {fraud_brr.mean():.4f}")
print(f"Legitimate accounts — mean retention rate: {legit_brr.mean():.4f}")
print(f"Fraud accounts with retention < 0.10: {(fraud_brr < 0.10).sum():,} ({(fraud_brr < 0.10).mean()*100:.1f}%)")
print(f"Legit accounts with retention < 0.10: {(legit_brr < 0.10).sum():,} ({(legit_brr < 0.10).mean()*100:.1f}%)")

=== FEATURE 2: BALANCE RETENTION RATE ===
Fraud accounts   — mean retention rate: 0.0091
Legitimate accounts — mean retention rate: 0.0541
Fraud accounts with retention < 0.10: 8,063 (98.2%)
Legit accounts with retention < 0.10: 2,507,542 (90.8%)


In [5]:
# FEATURE 3: FAN-OUT DEGREE
# How many unique destination accounts does this account send to?
# Mule accounts send to exactly 1 fixed destination — proven in EDA

account_stats['fan_out_degree'] = account_stats['unique_destinations']

# Validate
fraud_fo = account_stats[account_stats['is_fraud']==1]['fan_out_degree']
legit_fo = account_stats[account_stats['is_fraud']==0]['fan_out_degree']

print("=== FEATURE 3: FAN-OUT DEGREE ===")
print(f"Fraud accounts   — mean fan-out: {fraud_fo.mean():.4f}")
print(f"Legitimate accounts — mean fan-out: {legit_fo.mean():.4f}")
print(f"Fraud accounts with fan-out = 1:  {(fraud_fo==1).sum():,} ({(fraud_fo==1).mean()*100:.1f}%)")
print(f"Fraud accounts with fan-out <= 3: {(fraud_fo<=3).sum():,} ({(fraud_fo<=3).mean()*100:.1f}%)")
print(f"Legit accounts with fan-out = 1:  {(legit_fo==1).sum():,} ({(legit_fo==1).mean()*100:.1f}%)")

=== FEATURE 3: FAN-OUT DEGREE ===
Fraud accounts   — mean fan-out: 1.0019
Legitimate accounts — mean fan-out: 1.0006
Fraud accounts with fan-out = 1:  8,197 (99.8%)
Fraud accounts with fan-out <= 3: 8,213 (100.0%)
Legit accounts with fan-out = 1:  2,758,657 (99.9%)


In [6]:
# FEATURE 4: DORMANCY SPIKE SCORE
# How long was the account inactive before its first high-value transaction?
# Mule accounts sit dormant then suddenly activate

# Active window = last step minus first step
account_stats['active_window'] = (
    account_stats['last_active_step'] - account_stats['first_active_step']
)

# Dormancy spike score = high amount relative to short active window
# A high max_amount with a short active window = sudden activation
account_stats['dormancy_spike_score'] = account_stats.apply(
    lambda row: row['max_amount'] / (row['active_window'] + 1), axis=1
)

# Normalise to 0-1 range
max_score = account_stats['dormancy_spike_score'].quantile(0.99)
account_stats['dormancy_spike_score'] = (
    account_stats['dormancy_spike_score'] / max_score
).clip(0, 1)

# Validate
fraud_dss = account_stats[account_stats['is_fraud']==1]['dormancy_spike_score']
legit_dss = account_stats[account_stats['is_fraud']==0]['dormancy_spike_score']

print("=== FEATURE 4: DORMANCY SPIKE SCORE ===")
print(f"Fraud accounts   — mean dormancy spike: {fraud_dss.mean():.4f}")
print(f"Legitimate accounts — mean dormancy spike: {legit_dss.mean():.4f}")
print(f"Fraud accounts with score > 0.50: {(fraud_dss > 0.50).sum():,} ({(fraud_dss > 0.50).mean()*100:.1f}%)")
print(f"Legit accounts with score > 0.50: {(legit_dss > 0.50).sum():,} ({(legit_dss > 0.50).mean()*100:.1f}%)")

=== FEATURE 4: DORMANCY SPIKE SCORE ===
Fraud accounts   — mean dormancy spike: 0.3384
Legitimate accounts — mean dormancy spike: 0.1050
Fraud accounts with score > 0.50: 2,253 (27.4%)
Legit accounts with score > 0.50: 85,289 (3.1%)


In [7]:
# FEATURE 5: ROUND AMOUNT FLAG
# Are transactions suspiciously round numbers?
# Classic laundering signal — criminals often use round amounts

def round_amount_flag(amount):
    # Flag if divisible by 1000 with no remainder
    return 1 if amount % 1000 == 0 and amount > 0 else 0

# Apply to max transaction amount per account
account_stats['round_amount_flag'] = account_stats['max_amount'].apply(
    round_amount_flag
)

# Also check what % of an account's transactions are round
round_transactions = df_focus.copy()
round_transactions['is_round'] = round_transactions['amount'].apply(
    round_amount_flag
)
round_ratio = round_transactions.groupby('nameOrig')['is_round'].mean().reset_index()
round_ratio.columns = ['nameOrig', 'round_amount_ratio']
account_stats = account_stats.merge(round_ratio, on='nameOrig', how='left')

# Validate
fraud_raf = account_stats[account_stats['is_fraud']==1]['round_amount_ratio']
legit_raf = account_stats[account_stats['is_fraud']==0]['round_amount_ratio']

print("=== FEATURE 5: ROUND AMOUNT FLAG ===")
print(f"Fraud accounts   — mean round amount ratio: {fraud_raf.mean():.4f}")
print(f"Legitimate accounts — mean round amount ratio: {legit_raf.mean():.4f}")
print(f"Fraud accounts with round_amount_flag=1: {account_stats[account_stats['is_fraud']==1]['round_amount_flag'].sum():,}")
print(f"Legit accounts with round_amount_flag=1: {account_stats[account_stats['is_fraud']==0]['round_amount_flag'].sum():,}")

=== FEATURE 5: ROUND AMOUNT FLAG ===
Fraud accounts   — mean round amount ratio: 0.0349
Legitimate accounts — mean round amount ratio: 0.0011
Fraud accounts with round_amount_flag=1: 287
Legit accounts with round_amount_flag=1: 2,984


In [8]:
# FEATURE 6: TRANSACTION TYPE ENTROPY
# How varied are the transaction types for this account?
# Mule accounts use only TRANSFER or CASH_OUT repeatedly — very low entropy
# Legitimate users show more variety

from scipy.stats import entropy as scipy_entropy

def calc_entropy(group):
    type_counts = group['type'].value_counts(normalize=True)
    return scipy_entropy(type_counts)

type_entropy = df_focus.groupby('nameOrig').apply(
    calc_entropy
).reset_index()
type_entropy.columns = ['nameOrig', 'type_entropy']

account_stats = account_stats.merge(type_entropy, on='nameOrig', how='left')
account_stats['type_entropy'] = account_stats['type_entropy'].fillna(0)

# Validate
fraud_te = account_stats[account_stats['is_fraud']==1]['type_entropy']
legit_te = account_stats[account_stats['is_fraud']==0]['type_entropy']

print("=== FEATURE 6: TRANSACTION TYPE ENTROPY ===")
print(f"Fraud accounts   — mean entropy: {fraud_te.mean():.4f}")
print(f"Legitimate accounts — mean entropy: {legit_te.mean():.4f}")
print(f"Fraud accounts with entropy = 0: {(fraud_te==0).sum():,} ({(fraud_te==0).mean()*100:.1f}%)")
print(f"Legit accounts with entropy = 0: {(legit_te==0).sum():,} ({(legit_te==0).mean()*100:.1f}%)")

KeyboardInterrupt: 

In [9]:
# FEATURE 6: TRANSACTION TYPE ENTROPY — fast version
# Since we only have TRANSFER and CASH_OUT in df_focus,
# entropy is determined by the ratio of each type per account

type_counts = df_focus.groupby(['nameOrig', 'type']).size().unstack(fill_value=0)

# Add missing columns if either type is absent
for col in ['TRANSFER', 'CASH_OUT']:
    if col not in type_counts.columns:
        type_counts[col] = 0

# Calculate entropy manually — fast vectorised
total = type_counts.sum(axis=1)
p_transfer = type_counts['TRANSFER'] / total
p_cashout  = type_counts['CASH_OUT']  / total

# Entropy = -sum(p * log(p)), handle zeros
def safe_entropy(p):
    return np.where(p > 0, -p * np.log(p), 0)

type_entropy_vals = safe_entropy(p_transfer) + safe_entropy(p_cashout)
type_entropy_df = pd.DataFrame({
    'nameOrig': type_counts.index,
    'type_entropy': type_entropy_vals
})

account_stats = account_stats.merge(type_entropy_df, on='nameOrig', how='left')
account_stats['type_entropy'] = account_stats['type_entropy'].fillna(0)

# Validate
fraud_te = account_stats[account_stats['is_fraud']==1]['type_entropy']
legit_te = account_stats[account_stats['is_fraud']==0]['type_entropy']

print("=== FEATURE 6: TRANSACTION TYPE ENTROPY ===")
print(f"Fraud accounts   — mean entropy: {fraud_te.mean():.4f}")
print(f"Legitimate accounts — mean entropy: {legit_te.mean():.4f}")
print(f"Fraud accounts with entropy = 0:  {(fraud_te==0).sum():,} ({(fraud_te==0).mean()*100:.1f}%)")
print(f"Legit accounts with entropy = 0:  {(legit_te==0).sum():,} ({(legit_te==0).mean()*100:.1f}%)")

=== FEATURE 6: TRANSACTION TYPE ENTROPY ===
Fraud accounts   — mean entropy: 0.0008
Legitimate accounts — mean entropy: 0.0001
Fraud accounts with entropy = 0:  8,204 (99.9%)
Legit accounts with entropy = 0:  2,759,860 (100.0%)


In [10]:
# FEATURE 7: ACCOUNT AGE AT ACTIVATION
# How early in the 30-day period did this account first become active?
# Newly created mule accounts activate quickly after opening

# In PaySim, step = hour. First active step = proxy for account age.
# Lower first_active_step = account activated earlier in the period

account_stats['account_age_at_activation'] = account_stats['first_active_step']

# Normalise to 0-1
max_step = df_focus['step'].max()
account_stats['account_age_at_activation'] = (
    account_stats['account_age_at_activation'] / max_step
)

# Validate
fraud_aaa = account_stats[account_stats['is_fraud']==1]['account_age_at_activation']
legit_aaa = account_stats[account_stats['is_fraud']==0]['account_age_at_activation']

print("=== FEATURE 7: ACCOUNT AGE AT ACTIVATION ===")
print(f"Fraud accounts   — mean activation time: {fraud_aaa.mean():.4f} (normalised)")
print(f"Legitimate accounts — mean activation time: {legit_aaa.mean():.4f} (normalised)")
print(f"Fraud accounts activating in first 25% of period: {(fraud_aaa < 0.25).sum():,} ({(fraud_aaa < 0.25).mean()*100:.1f}%)")
print(f"Legit accounts activating in first 25% of period: {(legit_aaa < 0.25).sum():,} ({(legit_aaa < 0.25).mean()*100:.1f}%)")

=== FEATURE 7: ACCOUNT AGE AT ACTIVATION ===
Fraud accounts   — mean activation time: 0.4954 (normalised)
Legitimate accounts — mean activation time: 0.3251 (normalised)
Fraud accounts activating in first 25% of period: 2,110 (25.7%)
Legit accounts activating in first 25% of period: 967,976 (35.1%)


In [11]:
# FEATURE 8: COUNTERPARTY FAN-IN RATIO
# How many unique senders are depositing INTO this account?
# Mule accounts receive from many different sources (victims)
# then forward to one destination

fan_in = df_focus.groupby('nameDest')['nameOrig'].nunique().reset_index()
fan_in.columns = ['nameOrig', 'fan_in_degree']

account_stats = account_stats.merge(fan_in, on='nameOrig', how='left')
account_stats['fan_in_degree'] = account_stats['fan_in_degree'].fillna(0)

# Validate
fraud_fi = account_stats[account_stats['is_fraud']==1]['fan_in_degree']
legit_fi = account_stats[account_stats['is_fraud']==0]['fan_in_degree']

print("=== FEATURE 8: COUNTERPARTY FAN-IN RATIO ===")
print(f"Fraud accounts   — mean fan-in: {fraud_fi.mean():.4f}")
print(f"Legitimate accounts — mean fan-in: {legit_fi.mean():.4f}")
print(f"Fraud accounts with fan-in > 0: {(fraud_fi > 0).sum():,} ({(fraud_fi > 0).mean()*100:.1f}%)")
print(f"Legit accounts with fan-in > 0: {(legit_fi > 0).sum():,} ({(legit_fi > 0).mean()*100:.1f}%)")

=== FEATURE 8: COUNTERPARTY FAN-IN RATIO ===
Fraud accounts   — mean fan-in: 0.0000
Legitimate accounts — mean fan-in: 0.0014
Fraud accounts with fan-in > 0: 0 (0.0%)
Legit accounts with fan-in > 0: 686 (0.0%)


In [12]:
# FEATURE 9: TRANSACTION REGULARITY INDEX
# How regular are the time gaps between transactions?
# Automated mule operations produce machine-like regularity
# Human legitimate users show natural variance
# Low standard deviation of inter-transaction gaps = suspicious regularity

def calc_regularity(group):
    if len(group) < 2:
        return 0.0
    steps = sorted(group['step'].values)
    gaps = np.diff(steps)
    if len(gaps) == 0 or np.std(gaps) == 0:
        return 1.0  # perfectly regular = maximum suspicion
    # Coefficient of variation — lower = more regular = more suspicious
    cv = np.std(gaps) / (np.mean(gaps) + 1e-9)
    return 1 / (1 + cv)  # invert so high score = high regularity

# Only compute for accounts with more than 1 transaction
multi_txn = df_focus.groupby('nameOrig').filter(lambda x: len(x) > 1)

print("Computing transaction regularity index...")
regularity = multi_txn.groupby('nameOrig').apply(
    lambda g: calc_regularity(g)
).reset_index()
regularity.columns = ['nameOrig', 'regularity_index']

account_stats = account_stats.merge(regularity, on='nameOrig', how='left')
account_stats['regularity_index'] = account_stats['regularity_index'].fillna(0.5)

# Validate
fraud_ri = account_stats[account_stats['is_fraud']==1]['regularity_index']
legit_ri = account_stats[account_stats['is_fraud']==0]['regularity_index']

print("=== FEATURE 9: TRANSACTION REGULARITY INDEX ===")
print(f"Fraud accounts   — mean regularity: {fraud_ri.mean():.4f}")
print(f"Legitimate accounts — mean regularity: {legit_ri.mean():.4f}")
print(f"Fraud accounts with regularity > 0.70: {(fraud_ri > 0.70).sum():,} ({(fraud_ri > 0.70).mean()*100:.1f}%)")
print(f"Legit accounts with regularity > 0.70: {(legit_ri > 0.70).sum():,} ({(legit_ri > 0.70).mean()*100:.1f}%)")

Computing transaction regularity index...
=== FEATURE 9: TRANSACTION REGULARITY INDEX ===
Fraud accounts   — mean regularity: 0.5010
Legitimate accounts — mean regularity: 0.5003
Fraud accounts with regularity > 0.70: 16 (0.2%)
Legit accounts with regularity > 0.70: 1,758 (0.1%)


In [13]:
# FEATURE 10: MAX AMOUNT RATIO
# How large is the biggest transaction relative to the account's average?
# Mule accounts often have one very large transaction that is the actual
# criminal transfer, surrounded by smaller ones to avoid detection

account_stats['max_amount_ratio'] = account_stats.apply(
    lambda row: row['max_amount'] / (row['avg_amount'] + 1e-9), axis=1
)

# Normalise
max_ratio = account_stats['max_amount_ratio'].quantile(0.99)
account_stats['max_amount_ratio'] = (
    account_stats['max_amount_ratio'] / max_ratio
).clip(0, 1)

# Validate
fraud_mar = account_stats[account_stats['is_fraud']==1]['max_amount_ratio']
legit_mar = account_stats[account_stats['is_fraud']==0]['max_amount_ratio']

print("=== FEATURE 10: MAX AMOUNT RATIO ===")
print(f"Fraud accounts   — mean max ratio: {fraud_mar.mean():.4f}")
print(f"Legitimate accounts — mean max ratio: {legit_mar.mean():.4f}")
print(f"Fraud accounts with ratio > 0.50: {(fraud_mar > 0.50).sum():,} ({(fraud_mar > 0.50).mean()*100:.1f}%)")
print(f"Legit accounts with ratio > 0.50: {(legit_mar > 0.50).sum():,} ({(legit_mar > 0.50).mean()*100:.1f}%)")

=== FEATURE 10: MAX AMOUNT RATIO ===
Fraud accounts   — mean max ratio: 0.9981
Legitimate accounts — mean max ratio: 1.0000
Fraud accounts with ratio > 0.50: 8,197 (99.8%)
Legit accounts with ratio > 0.50: 2,760,417 (100.0%)


In [14]:
# COMPILE FINAL FEATURE MATRIX
feature_cols = [
    'nameOrig',
    'velocity_ratio',
    'balance_retention_rate', 
    'fan_out_degree',
    'dormancy_spike_score',
    'round_amount_flag',
    'round_amount_ratio',
    'type_entropy',
    'account_age_at_activation',
    'fan_in_degree',
    'regularity_index',
    'max_amount_ratio',
    'total_transactions',
    'avg_amount',
    'max_amount',
    'is_fraud'
]

feature_matrix = account_stats[feature_cols].copy()

# Check for nulls
print("=== FEATURE MATRIX SUMMARY ===")
print(f"Shape: {feature_matrix.shape}")
print(f"\nNull values per feature:")
print(feature_matrix.isnull().sum())
print(f"\nFraud accounts: {feature_matrix['is_fraud'].sum():,}")
print(f"Legitimate accounts: {(feature_matrix['is_fraud']==0).sum():,}")

# Fill any remaining nulls with 0
feature_matrix = feature_matrix.fillna(0)

# Save
feature_matrix.to_csv('feature_matrix.csv', index=False)
print("\nFeature matrix saved as feature_matrix.csv")
print(f"Final shape: {feature_matrix.shape}")

=== FEATURE MATRIX SUMMARY ===
Shape: (2768630, 16)

Null values per feature:
nameOrig                     0
velocity_ratio               0
balance_retention_rate       0
fan_out_degree               0
dormancy_spike_score         0
round_amount_flag            0
round_amount_ratio           0
type_entropy                 0
account_age_at_activation    0
fan_in_degree                0
regularity_index             0
max_amount_ratio             0
total_transactions           0
avg_amount                   0
max_amount                   0
is_fraud                     0
dtype: int64

Fraud accounts: 8,213
Legitimate accounts: 2,760,417

Feature matrix saved as feature_matrix.csv
Final shape: (2768630, 16)


In [15]:
# QUICK FEATURE CORRELATION WITH FRAUD LABEL
# Shows which features have strongest linear relationship with is_fraud
# before any ML model is applied

feature_cols_only = [
    'velocity_ratio', 'balance_retention_rate', 'fan_out_degree',
    'dormancy_spike_score', 'round_amount_flag', 'round_amount_ratio',
    'type_entropy', 'account_age_at_activation', 'fan_in_degree',
    'regularity_index', 'max_amount_ratio', 'total_transactions',
    'avg_amount', 'max_amount'
]

correlations = feature_matrix[feature_cols_only + ['is_fraud']].corr()['is_fraud'].drop('is_fraud')
correlations = correlations.abs().sort_values(ascending=False)

print("=== FEATURE CORRELATION WITH FRAUD LABEL ===")
print("(absolute correlation — higher = stronger linear signal)")
print()
for feat, corr in correlations.items():
    bar = "█" * int(corr * 50)
    print(f"{feat:<30} {corr:.4f}  {bar}")

=== FEATURE CORRELATION WITH FRAUD LABEL ===
(absolute correlation — higher = stronger linear signal)

dormancy_spike_score           0.0853  ████
avg_amount                     0.0707  ███
max_amount                     0.0707  ███
round_amount_ratio             0.0536  ██
round_amount_flag              0.0536  ██
account_age_at_activation      0.0486  ██
max_amount_ratio               0.0441  ██
velocity_ratio                 0.0136  
balance_retention_rate         0.0132  
type_entropy                   0.0034  
regularity_index               0.0028  
fan_out_degree                 0.0028  
total_transactions             0.0028  
fan_in_degree                  0.0006  


In [16]:
print("=" * 60)
print("WEEK 2 COMPLETE — FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("""
FEATURES BUILT: 10 engineered + 3 raw = 13 total input features

STRONG FEATURES (clear fraud/legit separation):
  - Dormancy Spike Score     : fraud mean 0.3384 vs legit 0.1050 (3.2x)
  - Round Amount Ratio       : fraud mean 0.0349 vs legit 0.0011 (31x)
  - Average Amount           : fraud accounts transact at higher values
  - Velocity Ratio           : fraud mean 0.9925 vs legit 0.9459

DATASET LIMITATION FEATURES (weak on PaySim, strong on real data):
  - Fan-Out Degree           : PaySim mostly single-destination accounts
  - Fan-In Degree            : Sender/receiver ID sets are disjoint
  - Type Entropy             : Most accounts use only one type
  - Regularity Index         : Most accounts have only 1 transaction

KEY INSIGHT:
  All correlations are low (max 0.085)
  No single feature is sufficient — validates multi-feature approach
  XGBoost will find non-linear combinations linear correlation misses
  Graph features in Phase 4 will provide the strongest signal

OUTPUT:
  feature_matrix.csv — 2,768,630 accounts x 16 columns
  All 8,213 fraud accounts retained
  Zero null values

NEXT: Phase 2 — ML Classification on this feature matrix
""")

WEEK 2 COMPLETE — FEATURE ENGINEERING SUMMARY

FEATURES BUILT: 10 engineered + 3 raw = 13 total input features

STRONG FEATURES (clear fraud/legit separation):
  - Dormancy Spike Score     : fraud mean 0.3384 vs legit 0.1050 (3.2x)
  - Round Amount Ratio       : fraud mean 0.0349 vs legit 0.0011 (31x)
  - Average Amount           : fraud accounts transact at higher values
  - Velocity Ratio           : fraud mean 0.9925 vs legit 0.9459

DATASET LIMITATION FEATURES (weak on PaySim, strong on real data):
  - Fan-Out Degree           : PaySim mostly single-destination accounts
  - Fan-In Degree            : Sender/receiver ID sets are disjoint
  - Type Entropy             : Most accounts use only one type
  - Regularity Index         : Most accounts have only 1 transaction

KEY INSIGHT:
  All correlations are low (max 0.085)
  No single feature is sufficient — validates multi-feature approach
  XGBoost will find non-linear combinations linear correlation misses
  Graph features in Phase 4

In [17]:
# FEATURE 11: STRUCTURING SCORE (SMURFING)
# How often does this account send amounts just below round thresholds?
# Criminals deliberately stay below reporting limits
# e.g. sending 9,999 instead of 10,000

def structuring_flag(amount):
    thresholds = [10000, 50000, 100000, 500000, 1000000]
    for t in thresholds:
        if (t - 500) <= amount < t:
            return 1
    return 0

df_focus_struct = df_focus.copy()
df_focus_struct['is_structured'] = df_focus_struct['amount'].apply(structuring_flag)

struct_ratio = df_focus_struct.groupby('nameOrig')['is_structured'].mean().reset_index()
struct_ratio.columns = ['nameOrig', 'structuring_score']

account_stats = account_stats.merge(struct_ratio, on='nameOrig', how='left')
account_stats['structuring_score'] = account_stats['structuring_score'].fillna(0)

# Validate
fraud_ss = account_stats[account_stats['is_fraud']==1]['structuring_score']
legit_ss = account_stats[account_stats['is_fraud']==0]['structuring_score']

print("=== FEATURE 11: STRUCTURING SCORE ===")
print(f"Fraud accounts   — mean structuring score: {fraud_ss.mean():.6f}")
print(f"Legitimate accounts — mean structuring score: {legit_ss.mean():.6f}")
print(f"Fraud accounts with score > 0: {(fraud_ss > 0).sum():,} ({(fraud_ss > 0).mean()*100:.2f}%)")
print(f"Legit accounts with score > 0: {(legit_ss > 0).sum():,} ({(legit_ss > 0).mean()*100:.2f}%)")

=== FEATURE 11: STRUCTURING SCORE ===
Fraud accounts   — mean structuring score: 0.002983
Legitimate accounts — mean structuring score: 0.004664
Fraud accounts with score > 0: 25 (0.30%)
Legit accounts with score > 0: 12,883 (0.47%)


Note: Feature computation here is batch processing for model training. In production (Phase 3), all features are maintained as incrementally updated account state in Redis, updated per transaction event via Apache Flink. Batch recomputation is only required for initial system deployment and weekly model retraining.

In [18]:
# FEATURE 12: TRANSACTION FREQUENCY ACCELERATION
# Is the account transacting faster over time?
# Mule accounts ramp up activity suddenly
# Compare transaction rate in first half vs second half of active window

def freq_acceleration(row):
    window = row['last_active_step'] - row['first_active_step']
    if window < 2:
        return 1.0  # single step = instant activation = suspicious
    mid = row['first_active_step'] + window / 2
    return min(row['total_transactions'] / (window + 1) * 2, 1.0)

account_stats['freq_acceleration'] = account_stats.apply(
    freq_acceleration, axis=1
)

fraud_fa = account_stats[account_stats['is_fraud']==1]['freq_acceleration']
legit_fa = account_stats[account_stats['is_fraud']==0]['freq_acceleration']

print("=== FEATURE 12: TRANSACTION FREQUENCY ACCELERATION ===")
print(f"Fraud accounts   — mean: {fraud_fa.mean():.4f}")
print(f"Legitimate accounts — mean: {legit_fa.mean():.4f}")
print(f"Ratio: {fraud_fa.mean()/legit_fa.mean():.2f}x")

=== FEATURE 12: TRANSACTION FREQUENCY ACCELERATION ===
Fraud accounts   — mean: 0.9983
Legitimate accounts — mean: 0.9994
Ratio: 1.00x


In [19]:
# FEATURE 13: AMOUNT VARIANCE SCORE
# How consistent are transaction amounts?
# Automated mule operations send very consistent amounts
# Low variance = high suspicion

amount_variance = df_focus.groupby('nameOrig')['amount'].agg(
    ['mean', 'std']
).reset_index()
amount_variance.columns = ['nameOrig', 'amount_mean', 'amount_std']
amount_variance['amount_std'] = amount_variance['amount_std'].fillna(0)

# Coefficient of variation — lower = more consistent = more suspicious
# Invert so high score = high suspicion
amount_variance['amount_variance_score'] = amount_variance.apply(
    lambda row: 1 / (1 + (row['amount_std'] / (row['amount_mean'] + 1e-9))),
    axis=1
)

account_stats = account_stats.merge(
    amount_variance[['nameOrig', 'amount_variance_score']],
    on='nameOrig', how='left'
)
account_stats['amount_variance_score'] = account_stats['amount_variance_score'].fillna(1.0)

fraud_avs = account_stats[account_stats['is_fraud']==1]['amount_variance_score']
legit_avs = account_stats[account_stats['is_fraud']==0]['amount_variance_score']

print("=== FEATURE 13: AMOUNT VARIANCE SCORE ===")
print(f"Fraud accounts   — mean: {fraud_avs.mean():.4f}")
print(f"Legitimate accounts — mean: {legit_avs.mean():.4f}")
print(f"Ratio: {fraud_avs.mean()/legit_avs.mean():.2f}x")

=== FEATURE 13: AMOUNT VARIANCE SCORE ===
Fraud accounts   — mean: 0.9993
Legitimate accounts — mean: 0.9998
Ratio: 1.00x


In [20]:
# FEATURE 14: SESSION BURST SCORE
# What is the maximum number of transactions in any single hour?
# Mule operators working in batches produce hourly bursts
# High burst = suspicious

burst = df_focus.groupby(['nameOrig', 'step']).size().reset_index()
burst.columns = ['nameOrig', 'step', 'txn_per_step']

max_burst = burst.groupby('nameOrig')['txn_per_step'].max().reset_index()
max_burst.columns = ['nameOrig', 'session_burst_score']

account_stats = account_stats.merge(max_burst, on='nameOrig', how='left')
account_stats['session_burst_score'] = account_stats['session_burst_score'].fillna(1)

# Normalise
max_burst_val = account_stats['session_burst_score'].quantile(0.99)
account_stats['session_burst_score'] = (
    account_stats['session_burst_score'] / max_burst_val
).clip(0, 1)

fraud_sbs = account_stats[account_stats['is_fraud']==1]['session_burst_score']
legit_sbs = account_stats[account_stats['is_fraud']==0]['session_burst_score']

print("=== FEATURE 14: SESSION BURST SCORE ===")
print(f"Fraud accounts   — mean: {fraud_sbs.mean():.4f}")
print(f"Legitimate accounts — mean: {legit_sbs.mean():.4f}")
print(f"Ratio: {fraud_sbs.mean()/legit_sbs.mean():.2f}x")
print(f"Fraud accounts with burst > 0.50: {(fraud_sbs > 0.50).sum():,} ({(fraud_sbs > 0.50).mean()*100:.1f}%)")

=== FEATURE 14: SESSION BURST SCORE ===
Fraud accounts   — mean: 1.0000
Legitimate accounts — mean: 1.0000
Ratio: 1.00x
Fraud accounts with burst > 0.50: 8,213 (100.0%)


In [21]:
# FEATURE 15: DORMANCY RATIO
# What proportion of the account's active window has zero transactions?
# Mule accounts are either fully active or completely dormant
# High dormancy ratio = suspicious

# Total steps in active window
account_stats['active_window'] = (
    account_stats['last_active_step'] - account_stats['first_active_step'] + 1
)

# Steps with actual transactions
steps_active = df_focus.groupby('nameOrig')['step'].nunique().reset_index()
steps_active.columns = ['nameOrig', 'steps_with_transactions']

account_stats = account_stats.merge(steps_active, on='nameOrig', how='left')
account_stats['steps_with_transactions'] = account_stats['steps_with_transactions'].fillna(1)

# Dormancy ratio = proportion of window with NO transactions
account_stats['dormancy_ratio'] = account_stats.apply(
    lambda row: 1 - (row['steps_with_transactions'] / row['active_window'])
    if row['active_window'] > 0 else 0, axis=1
).clip(0, 1)

fraud_dr = account_stats[account_stats['is_fraud']==1]['dormancy_ratio']
legit_dr = account_stats[account_stats['is_fraud']==0]['dormancy_ratio']

print("=== FEATURE 15: DORMANCY RATIO ===")
print(f"Fraud accounts   — mean: {fraud_dr.mean():.4f}")
print(f"Legitimate accounts — mean: {legit_dr.mean():.4f}")
print(f"Ratio: {fraud_dr.mean()/(legit_dr.mean()+1e-9):.2f}x")
print(f"Fraud accounts with ratio > 0.50: {(fraud_dr > 0.50).sum():,} ({(fraud_dr > 0.50).mean()*100:.1f}%)")

=== FEATURE 15: DORMANCY RATIO ===
Fraud accounts   — mean: 0.0018
Legitimate accounts — mean: 0.0006
Ratio: 3.04x
Fraud accounts with ratio > 0.50: 15 (0.2%)


In [22]:
# FEATURE 16: ONBOARDING VELOCITY
# How quickly did the account reach its maximum transaction amount?
# Mule accounts hit peak amounts immediately after activation
# Legitimate accounts ramp up gradually

account_stats['onboarding_velocity'] = account_stats.apply(
    lambda row: 1.0 if row['active_window'] <= 1
    else 1 - (row['first_active_step'] / (row['last_active_step'] + 1e-9)),
    axis=1
).clip(0, 1)

fraud_ov = account_stats[account_stats['is_fraud']==1]['onboarding_velocity']
legit_ov = account_stats[account_stats['is_fraud']==0]['onboarding_velocity']

print("=== FEATURE 16: ONBOARDING VELOCITY ===")
print(f"Fraud accounts   — mean: {fraud_ov.mean():.4f}")
print(f"Legitimate accounts — mean: {legit_ov.mean():.4f}")
print(f"Ratio: {fraud_ov.mean()/(legit_ov.mean()+1e-9):.2f}x")

=== FEATURE 16: ONBOARDING VELOCITY ===
Fraud accounts   — mean: 0.9992
Legitimate accounts — mean: 0.9997
Ratio: 1.00x


In [24]:
# UPDATED FEATURE MATRIX WITH ALL 16 FEATURES
feature_cols_updated = [
    'nameOrig',
    'velocity_ratio',
    'balance_retention_rate',
    'fan_out_degree',
    'dormancy_spike_score',
    'round_amount_flag',
    'round_amount_ratio',
    'type_entropy',
    'account_age_at_activation',
    'fan_in_degree',
    'regularity_index',
    'max_amount_ratio',
    'total_transactions',
    'avg_amount',
    'max_amount',
    'structuring_score',
    'freq_acceleration',
    'amount_variance_score',
    'session_burst_score',
    'dormancy_ratio',
    'onboarding_velocity',
    'is_fraud'
]

feature_matrix_v2 = account_stats[feature_cols_updated].copy()
feature_matrix_v2 = feature_matrix_v2.fillna(0)

print("=== UPDATED FEATURE MATRIX ===")
print(f"Shape: {feature_matrix_v2.shape}")
print(f"Features: {len(feature_cols_updated) - 2}")
print(f"Fraud accounts: {feature_matrix_v2['is_fraud'].sum():,}")
print(f"Null values: {feature_matrix_v2.isnull().sum().sum()}")

# Updated correlation table — handle NaN
feature_only = [c for c in feature_cols_updated if c not in ['nameOrig','is_fraud']]
correlations = feature_matrix_v2[feature_only + ['is_fraud']].corr()['is_fraud'].drop('is_fraud')
correlations = correlations.abs().fillna(0).sort_values(ascending=False)

print("\n=== UPDATED FEATURE CORRELATIONS ===")
for feat, corr in correlations.items():
    bar = "█" * int(corr * 50)
    print(f"{feat:<30} {corr:.4f}  {bar}")

feature_matrix_v2.to_csv('feature_matrix_v2.csv', index=False)
print("\nSaved as feature_matrix_v2.csv")

=== UPDATED FEATURE MATRIX ===
Shape: (2768630, 22)
Features: 20
Fraud accounts: 8,213
Null values: 0

=== UPDATED FEATURE CORRELATIONS ===
dormancy_spike_score           0.0853  ████
avg_amount                     0.0707  ███
max_amount                     0.0707  ███
round_amount_ratio             0.0536  ██
round_amount_flag              0.0536  ██
account_age_at_activation      0.0486  ██
max_amount_ratio               0.0441  ██
velocity_ratio                 0.0136  
balance_retention_rate         0.0132  
type_entropy                   0.0034  
regularity_index               0.0028  
fan_out_degree                 0.0028  
total_transactions             0.0028  
dormancy_ratio                 0.0028  
freq_acceleration              0.0027  
amount_variance_score          0.0027  
onboarding_velocity            0.0018  
structuring_score              0.0013  
fan_in_degree                  0.0006  
session_burst_score            0.0000  

Saved as feature_matrix_v2.csv
